# PageForge v2 — 节点流程模拟

不启动服务，直接调用各节点函数，模拟完整 graph 流程。

**测试场景**：`"帮我做一个极简风格的 Todo 应用"` (code_gen → 完整6节点流程)

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
os.chdir('/Users/fangyan/WorkBuddy/20260416190835/agent-projects-100/projects/pageforge/backend')
print(f'cwd: {os.getcwd()}')
print(f'Python: {sys.version}')

## Step 0: 注入 mock 事件发射器

不启动 SSE 服务，用全局 `set_event_emitter` 把事件打印到控制台。

In [ ]:
from app.graph.nodes.event_emitter import set_event_emitter

# 全局事件日志
event_log = []

def mock_emitter(event_type: str, data: dict):
    """把 SSE 事件打印出来"""
    import time
    ts = time.strftime('%H:%M:%S')
    entry = {'ts': ts, 'type': event_type, 'data': data}
    event_log.append(entry)
    # 简化打印
    short_data = {k: (v[:60] + '...' if isinstance(v, str) and len(v) > 60 else v) 
                  for k, v in data.items()}
    print(f'  📡 [{ts}] {event_type}  {short_data}')

set_event_emitter(mock_emitter)
print('✅ 事件发射器已注入')

## Step 1: 构造初始 State

模拟用户输入 → 初始 AgentState

In [ ]:
import uuid

user_message = "帮我做一个极简风格的 Todo 应用，支持添加、删除、标记完成"
session_id = str(uuid.uuid4())[:8]

initial_state = {
    'user_message': user_message,
    'session_id': session_id,
    'base_html': '',
    'task_list': [],
    'current_html': '',
    'validation_errors': [],
    'iteration_count': 0,
    'fix_count': 0,
    'response_message': '',
    'output_html': None,
    'output_version': 0,
    'is_complete': False,
    'project_type': None,
    'files': [],
    'project_id': None,
    'install_status': '',
    'dev_server_url': None,
    'ui_style': None,
    'intent': None,
    'confidence': 0.0,
    'tags': [],
    'mode': None,
    'complexity': None,
    'model_strategy': {},
    'thought_summary': '',
    'plan_steps': [],
    'ui_style_config': '',
    'status': '',
}

print(f'用户消息: {user_message}')
print(f'Session: {session_id}')
print(f'State keys: {list(initial_state.keys())}')
print('\n--- 开始模拟 ---')

## Step 2: intent_router — 意图识别

第一个节点，分析用户输入 → intent/model_strategy

In [ ]:
print('=' * 60)
print('🔍 Node 1: intent_router')
print('=' * 60)

from app.graph.nodes.intent_router import intent_router

state_after_intent = intent_router(initial_state)

print(f'\n📊 结果:')
print(f'   intent      = {state_after_intent.get("intent")}')
print(f'   confidence  = {state_after_intent.get("confidence")}')
print(f'   tags        = {state_after_intent.get("tags")}')
print(f'   mode        = {state_after_intent.get("mode")}')
print(f'   complexity  = {state_after_intent.get("complexity")}')
print(f'   ui_style    = {state_after_intent.get("ui_style")}')
print(f'   model_strategy = {state_after_intent.get("model_strategy")}')

## Step 3: thinking — 思维链

LLM 思考过程 → thought_summary

In [ ]:
print('\n' + '=' * 60)
print('🧠 Node 2: thinking')
print('=' * 60)

from app.graph.nodes.thinking import thinking_node

state_after_thinking = thinking_node(state_after_intent)

print(f'\n📊 结果:')
print(f'   thought_summary = {state_after_thinking.get("thought_summary", "")[:120]}...')
print(f'   新增 SSE 事件数: {len([e for e in event_log if "thinking" in e["type"]])}')

## Step 4: plan — 计划制定

LLM 生成执行计划 → plan_steps

In [ ]:
print('\n' + '=' * 60)
print('📋 Node 3: plan')
print('=' * 60)

from app.graph.nodes.plan import plan_node

state_after_plan = plan_node(state_after_thinking)

print(f'\n📊 结果:')
steps = state_after_plan.get('plan_steps', [])
print(f'   共 {len(steps)} 个步骤:')
for s in steps:
        print(f'   Step {s["id"]}: {s["label"]} (type={s.get("type", "?")})')
print(f'   新增 SSE 事件数: {len([e for e in event_log if "plan" in e["type"]])}')

## Step 5: style_picker — 风格选择

根据 ui_style 选择风格配置

In [ ]:
print('\n' + '=' * 60)
print('🎨 Node 4: style_picker')
print('=' * 60)

from app.graph.nodes.style_picker import style_picker_node

state_after_style = style_picker_node(state_after_plan)

print(f'\n📊 结果:')
print(f'   ui_style        = {state_after_style.get("ui_style")}')
print(f'   ui_style_config 长度 = {len(state_after_style.get("ui_style_config", ""))} chars')
print(f'   新增 SSE 事件数: {len([e for e in event_log if "style" in e["type"]])}')

## Step 6: code_gen — 代码生成

生成项目文件 → files[]

In [ ]:
print('\n' + '=' * 60)
print('⚡ Node 5: code_gen')
print('=' * 60)

from app.graph.nodes.code_gen import code_gen_node

state_after_code = code_gen_node(state_after_style)

print(f'\n📊 结果:')
files = state_after_code.get('files', [])
print(f'   生成 {len(files)} 个文件:')
for f in files:
        print(f'   📄 {f["path"]} ({f.get("language", "?")})')
print(f'   project_id   = {state_after_code.get("project_id")}')
print(f'   install_status = {state_after_code.get("install_status")}')
print(f'   新增 SSE 事件数: {len([e for e in event_log if "file" in e["type"] or "tool" in e["type"]])}')

## Step 7: reply — 最终回复

LLM 生成总结回复 → response_message

In [ ]:
print('\n' + '=' * 60)
print('💬 Node 6: reply')
print('=' * 60)

from app.graph.nodes.reply import reply_node

final_state = reply_node(state_after_code)

print(f'\n📊 结果:')
print(f'   response_message = {final_state.get("response_message", "")[:200]}')
print(f'   is_complete      = {final_state.get("is_complete")}')
print(f'   新增 SSE 事件数: {len([e for e in event_log if "text" in e["type"]])}')

## 📊 全流程事件统计

汇总所有 SSE 事件

In [ ]:
print('\n' + '=' * 60)
print('📊 全流程 SSE 事件统计')
print('=' * 60)

from collections import Counter

# 按类型分组
event_types = Counter(e['type'] for e in event_log)
print(f'\n共 {len(event_log)} 个事件:\n')

for etype, count in sorted(event_types.items()):
    print(f'  {etype:<25} x{count}')

print('\n--- 事件时间线 ---')
for e in event_log:
    data_preview = str(e['data'])[:80]
    print(f'  [{e["ts"]}] {e["type"]:<25} {data_preview}')

## 🔀 场景2: chat 意图（走 shortcut 路径）

测试 `intent_router → reply` 的快捷路径

In [ ]:
print('\n' + '=' * 60)
print('🔀 场景2: chat 意图 (shortcut 路径)')
print('=' * 60)

# 清空事件日志
event_log.clear()

chat_state = {
    **initial_state,
    'user_message': '你好，PageForge 是什么？',
}

print(f'用户消息: {chat_state["user_message"]}')
print('\n--- Step 1: intent_router ---')

chat_after_intent = intent_router(chat_state)
print(f'intent = {chat_after_intent.get("intent")}')

# chat 意图 → 直接走 reply，跳过 thinking/plan/style_picker/code_gen
print('\n--- Step 2: reply (shortcut) ---')
chat_final = reply_node(chat_after_intent)
print(f'\nresponse_message = {chat_final.get("response_message", "")}')
print(f'is_complete = {chat_final.get("is_complete")}')
print(f'\n共 {len(event_log)} 个 SSE 事件')